# Analyze Segmentation Label Classes in Dataset

This notebook analyzes PNG label masks in your dataset to find and summarize important information for conversion to YOLOv8 segmentation format.

## It will:
- List all unique class indices found in the label masks.
- Print the number of images and labels.
- Print the frequency (number of pixels) for each class across the entire dataset.
- Warn if there are image-label mismatches or shape mismatches.
- Print per-image and global statistics for quick inspection.

You can use this to verify your class indices and dataset before doing a YOLO conversion.


In [1]:
# --- Imports ---
import os
from glob import glob
import numpy as np
from PIL import Image


In [7]:
# --- Directory Paths (set as needed) ---
image_dir = 'D:/TomatoBotv2/Dataset/Crop Lane Segmentation/Segmentation Data/auto-labels/images'
label_dir = 'D:/TomatoBotv2/Dataset/Crop Lane Segmentation/Segmentation Data/auto-labels\labels'

In [8]:
# --- Gather file lists ---
image_files = sorted(glob(os.path.join(image_dir, '*')))
label_files = sorted(glob(os.path.join(label_dir, '*')))
print(f"Number of images: {len(image_files)}")
print(f"Number of labels: {len(label_files)}")

Number of images: 1163
Number of labels: 1163


In [9]:
# --- Check image-label correspondence ---
image_basenames = set(os.path.splitext(os.path.basename(f))[0] for f in image_files)
label_basenames = set(os.path.splitext(os.path.basename(f))[0] for f in label_files)

missing_labels = image_basenames - label_basenames
missing_images = label_basenames - image_basenames

if missing_labels:
    print(f"Warning: {len(missing_labels)} images without labels:")
    print(sorted(list(missing_labels))[:10], '...')
else:
    print("All images have corresponding labels.")

if missing_images:
    print(f"Warning: {len(missing_images)} labels without images:")
    print(sorted(list(missing_images))[:10], '...')
else:
    print("All labels have corresponding images.")

All images have corresponding labels.
All labels have corresponding images.


In [10]:
# --- Analyze class indices in all label masks ---
all_classes = set()
class_pixel_counts = dict()
shapes = set()

for label_path in label_files:
    label = np.array(Image.open(label_path))
    shapes.add(label.shape)
    unique, counts = np.unique(label, return_counts=True)
    for u, c in zip(unique, counts):
        all_classes.add(u)
        class_pixel_counts[u] = class_pixel_counts.get(u, 0) + c

print(f"\nUnique class indices present in all masks: {sorted(all_classes)}")
print(f"Total number of unique classes: {len(all_classes)}")

print("\nPixel count per class (across all images):")
for cls in sorted(class_pixel_counts.keys()):
    print(f"  Class {cls}: {class_pixel_counts[cls]:,} pixels")


Unique class indices present in all masks: [np.uint8(0), np.uint8(2), np.uint8(3), np.uint8(4), np.uint8(5), np.uint8(6), np.uint8(7), np.uint8(8)]
Total number of unique classes: 8

Pixel count per class (across all images):
  Class 0: 1,038,419,352 pixels
  Class 2: 2,561,150,961 pixels
  Class 3: 1,179,487,786 pixels
  Class 4: 2,673,577 pixels
  Class 5: 920,520 pixels
  Class 6: 4,148,020 pixels
  Class 7: 3,302,185 pixels
  Class 8: 10,889,279 pixels


In [11]:
# --- Check if all label shapes are consistent ---
if len(shapes) == 1:
    print(f"All labels have the same shape: {list(shapes)[0]}")
else:
    print(f"Warning: Labels have varying shapes: {shapes}")

## Next Steps
- Use the class indices found above to build your YOLO class list (data.yaml).
- If you want to map class indices to names, do so after reviewing this output.
- For YOLOv8 segmentation, you'll need to extract polygons for each class from each label mask. This can be implemented in the next notebook step.


In [12]:
import os
from glob import glob
import numpy as np
from PIL import Image
import cv2

# Paths (edit as needed)
image_dir = 'images/'   # directory with your images (jpg, png, etc.)
mask_dir = 'labels/'    # directory with your masks (png, same base names as images)
output_label_dir = 'yolo_labels/'  # output directory for YOLO txt files

image_dir = 'D:/TomatoBotv2/Dataset/Crop Lane Segmentation/Segmentation Data/auto-labels/images'
mask_dir = 'D:/TomatoBotv2/Dataset/Crop Lane Segmentation/Segmentation Data/auto-labels/labels'
output_label_dir = 'D:/TomatoBotv2/Dataset/Crop Lane Segmentation/Segmentation Data/auto-labels/yolo_labels'

os.makedirs(output_label_dir, exist_ok=True)

# Class indices (should match your data.yaml)
class_indices = list(range(9))  # 0-8

def mask_to_polygons(mask, class_id):
    # Returns a list of polygons for the given class (each polygon is a Nx2 numpy array)
    mask_bin = (mask == class_id).astype(np.uint8)
    contours, _ = cv2.findContours(mask_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polygons = []
    for contour in contours:
        if len(contour) >= 6:  # at least 3 points, YOLO wants at least 6 numbers
            polygons.append(contour.reshape(-1, 2))
    return polygons

def normalize_points(points, width, height):
    return [(x / width, y / height) for x, y in points]

# Loop through masks
mask_files = sorted(glob(os.path.join(mask_dir, '*.png')))
for mask_path in mask_files:
    base = os.path.splitext(os.path.basename(mask_path))[0]
    # Find image to get size
    for ext in ('.jpg', '.jpeg', '.JPG', '.JPEG', '.png'):
        img_path = os.path.join(image_dir, base + ext)
        if os.path.exists(img_path):
            break
    else:
        print(f"Image for {mask_path} not found!")
        continue
    img = Image.open(img_path)
    width, height = img.size
    mask = np.array(Image.open(mask_path))

    out_lines = []
    for class_id in class_indices:
        polygons = mask_to_polygons(mask, class_id)
        for poly in polygons:
            poly_norm = normalize_points(poly, width, height)
            # Flatten and join
            coords = []
            for x, y in poly_norm:
                coords.extend([f"{x:.6f}", f"{y:.6f}"])
            line = f"{class_id} " + " ".join(coords)
            out_lines.append(line)
    # Write YOLO label file
    with open(os.path.join(output_label_dir, base + '.txt'), 'w') as f:
        for line in out_lines:
            f.write(line + '\n')